In [6]:
#imports
import time
import pandas as pd
import pymongo
from pymongo import MongoClient, DESCENDING, TEXT

In [7]:
client = MongoClient("mongodb://localhost:27017/")
db = client["454"]


In [8]:
#drop indexes for reproducibility
db["All_beauty_meta_cleaned"].drop_indexes()
db["All_beauty_review_cleaned"].drop_indexes()

In [9]:
#indexes for scalability
db["All_beauty_meta_cleaned"].create_index("parent_asin", unique=True)
    
db["All_beauty_review_cleaned"].create_index([("parent_asin", 1), ("timestamp_convert", -1)])
    
db["All_beauty_review_cleaned"].create_index([("user_id", 1), ("timestamp_convert", -1)])
    
db["All_beauty_meta_cleaned"].create_index([
        ("title", TEXT), 
        ("description", TEXT)
    ], name="KeywordSearchIndex")

'KeywordSearchIndex'

In [10]:
#get the timings in ms
def benchmark_query(query_name, query_func, *args):
    start_time = time.time()
    result = query_func(*args)
    end_time = time.time()
    duration = (end_time - start_time) * 1000
    return {"Query": query_name, "Execution Time (ms)": round(duration, 4)}

In [11]:
#queries
def query_1(asin):
    return db["All_beauty_meta_cleaned"].find_one({"parent_asin": asin}, {"_id": 0, "title": 1, "description": 1})


def query_2(asin):
    return db["All_beauty_review_cleaned"].find({"parent_asin": asin},
                                         {"_id": 0, "rating": 1, "title": 1, "text": 1, "timestamp_convert": 1}).sort("timestamp_convert", DESCENDING).limit(10)


def query_5(asin):
    pipeline = [
        #find all reviews by this specific product
        { "$match": { "parent_asin": asin } },
        
        #look in both collections and get the ratings
        {
            "$lookup": {
                "from": "All_beauty_review_cleaned",   
                "localField": "parent_asin",      
                "foreignField": "parent_asin",    
                "as": "raw_ratings"               
            }
        },
        
        #create a new column to answer the query
        {
            "$addFields": {
                "rating_distribution": {
                    "$arrayToObject": {
                        "$map": { #loop start
                            "input": [1, 2, 3, 4, 5],
                            "as": "star",
                            "in": {
                                "k": { "$toString": "$$star" }, #field name 
                                "v": { #value for each fields of number of stars per field
                                    "$size": {
                                        "$filter": {
                                            "input": "$raw_ratings",
                                            "as": "r",
                                            "cond": { "$eq": ["$$r.rating", "$$star"] }
                                        }
                                    }
                                }
                            }
                        }
                    }
                }
            }
        },
        
        #only output what we want to answer the query
        {
            "$project": {
                "_id": 0,
                "parent_asin": 1,
                "average_rating": 1,
                "rating_number": 1,
                "rating_distribution": 1
            }
        }
    ]
    
    return list(db["All_beauty_meta_cleaned"].aggregate(pipeline))

def query_4(user_id):
    pipeline = [
        #find all reviews by this specific user
        { "$match": { "user_id": user_id } },

        { "$sort": { "timestamp_convert": DESCENDING } },

        #get data from both collections since both are needed
        {
            "$lookup": {
                "from": "All_beauty_meta_cleaned",    
                "localField": "parent_asin",   
                "foreignField": "parent_asin", 
                "as": "product_details"
            }
        },
        
        #flatten the joined product data
        { "$unwind": "$product_details" },
        
        #show only what we need - able equivalent
        {
            "$project": {
                "_id": 0,
                "user_id": 1,
                "rating": 1,
                "timestamp": 1,
                "product_title": "$product_details.title",
                "parent_asin": 1
            }
        }
        
    ]
    
    return list(db["All_beauty_review_cleaned"].aggregate(pipeline))

def query_3(keyword):
    return db["All_beauty_meta_cleaned"].find(
        {"$text": {"$search": keyword}},
        {"_id": 0, "title": 1, "parent_asin": 1, "description": 1}
    ).limit(10)

In [12]:
#############################################################
#test
results = []
test_asin = "B088BDFDHX"
test_user = "AFSKPY37N3C43SOI5IEXEK5JSIYA"
test_keyword = "hair"

results.append(benchmark_query("Query 1", query_1, test_asin))
results.append(benchmark_query("Query 2", query_2, test_asin))
results.append(benchmark_query("Query 3", query_3, test_keyword))
results.append(benchmark_query("Query 4", query_4, test_user))
results.append(benchmark_query("Query 5", query_5, test_asin))

In [13]:
df = pd.DataFrame(results)
print(df)

     Query  Execution Time (ms)
0  Query 1              21.2259
1  Query 2               0.0930
2  Query 3               0.0410
3  Query 4              43.4337
4  Query 5              44.9960


In [14]:
print(query_1(test_asin))

for review in query_2(test_asin):
    print(f"Date: {review['timestamp_convert']}")
    print(f"Rating: {review['rating']} Stars")
    print(f"Title: {review['title']}")
    print(f"Review: {review['text'][:100]}...")

for product in query_3(test_keyword):
    print(f"ASIN: {product['parent_asin']}")
    print(f"Title: {product['title']}")

print(query_4(test_user))
print(query_5(test_asin))

{'description': '', 'title': 'bfasu hair comb detangling anti static wooden comb woman men wide tooth wood comb curly hair 100 natural sandalwood'}
Date: 2021-08-11T12:57:39.732+01:00
Rating: 1 Stars
Title: crack break hair
Review: fine week crack appeared teeth started pulling breaking pulling hair...
Date: 2021-08-04T00:08:24.502+01:00
Rating: 5 Stars
Title: sturdy good quality wood
Review: work best scalp massager non static well use daily combing hair...
Date: 2021-07-02T15:17:14.249+01:00
Rating: 5 Stars
Title: wood best
Review: love comb natural curly hair doesn pull snag curl...
Date: 2021-06-27T01:01:41.665+01:00
Rating: 5 Stars
Title: magical comb
Review: comb magic able comb natural hair fraction time took love...
Date: 2021-05-30T02:44:11.526+01:00
Rating: 5 Stars
Title: good comb natural hair
Review: comb actually lifted hair like real comb breakage limited...
Date: 2021-04-01T01:48:28.211+01:00
Rating: 1 Stars
Title: disappointing loss money
Review: comb nice solid hold go

In [15]:
def export_product():
    collection = db["All_beauty_meta_cleaned"]

    #choose only the ones needed
    cursor = collection.find(
        {}, 
        {"_id": 0, "parent_asin": 1, "title": 1, "description": 1,"Item Form": 1,"Brand":1,"features_string":1}
    )
    
    df = pd.DataFrame(list(cursor))
    
    #convert list of strings to a single string
    df['Item Form'] = df['Item Form'].apply(
        lambda x: " ".join(x) if isinstance(x, list) else (str(x) if pd.notnull(x) else ""))
    
    df['Brand'] = df['Brand'].apply(
        lambda x: " ".join(x) if isinstance(x, list) else (str(x) if pd.notnull(x) else ""))
    
    df['features_string'] = df['features_string'].apply(
        lambda x: " ".join(x) if isinstance(x, list) else (str(x) if pd.notnull(x) else ""))

    df['combined'] = df['title'].fillna('') + " " + df['description'].fillna('') + " " + df['Brand'].fillna('') + " " + df['Item Form'].fillna('') + " " + df['features_string'].fillna('')

    df['Small'] = df['title'].fillna('') + " " + df['description'].fillna('')
    
    df.to_parquet("product_vec.parquet", index=False)
    
export_product()

In [16]:
def export_user():
    collection = db["All_beauty_review_cleaned"]
    
    #pipeline was the only way it worked to connect the two collections
    pipeline = [
        #connect to meta data collection
        {
            "$lookup": {
                "from": "All_beauty_meta_cleaned",
                "localField": "parent_asin",
                "foreignField": "parent_asin",
                "as": "product_details"
            }
        },
        #get title from joined data
        {
            "$set": {
                "product_title": { "$arrayElemAt": ["$product_details.title", 0] }
            }
        },
        #group by user and collect all titles they've interacted with
        {
            "$group": {
                "_id": "$user_id", 
                "combined_interests": { "$push": "$product_title" },
                "item_ids": { "$push": "$parent_asin" }
            }
        }
    ]
    #get text
    user_data = list(collection.aggregate(pipeline))
    df_users = pd.DataFrame(user_data)

    #create the item_string from titles for similarity
    df_users['item_string'] = df_users['combined_interests'].apply(
        lambda x: " ".join([str(i) for i in x if i])
    )

    df_users.to_parquet("user_vec.parquet", index=False)
    print(f"Success! {len(df_users)} user profiles exported.")

export_user()

Success! 631986 user profiles exported.
